### 结构化输出

In [1]:
import os
from langchain_openai import ChatOpenAI
#获取api_key
api_key = os.getenv('ARK_API_KEY')
model = ChatOpenAI(
    openai_api_base="https://ark.cn-beijing.volces.com/api/v3",
    openai_api_key=api_key,	# app_key
    model_name="doubao-seed-1-6-flash-250828",	# 您想使用的特定模型的名称或标识符。
    max_tokens=5000, #限制响应中的令牌总数，有效控制输出长度。
    temperature= 0.7,  #控制模型输出的随机性。值越高，响应越具创造性；值越低，响应越确定性。
    timeout=30, #模型的响应时间s
)
response = model.invoke("介绍一下你自己")

In [6]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from dataclasses import dataclass
from typing_extensions import TypedDict

# pydantic model
class ContactInfo1(BaseModel):
    """一个人的联系方式"""
    name: str = Field(description="该人的姓名")
    email: str = Field(description="该人的电子邮件地址")
    phone: str = Field(description="该人的电话号码")

# dataclass
@dataclass
class ContactInfo2:
    """一个人的联系方式"""
    name: str # 该人的姓名
    email: str # 该人的电子邮件地址
    phone: str # 该人的电话号码

class ContactInfo3(TypedDict):
    """一个人的联系信息。"""
    name: str # 该人的姓名
    email: str # 该人的电子邮件地址
    phone: str # 该人的电话号码

# JSON schema
contact_info_schema = {
    "type": "object",
    "description": "一个人的联系信息。",
    "properties": {
        "name": {"type": "string", "description": "该人的姓名"},
        "email": {"type": "string", "description": "该人的电子邮件地址"},
        "phone": {"type": "string", "description": "该人的电话号码"}
    },
    "required": ["name", "email", "phone"]
}

agent = create_agent(
    model,
    response_format= ContactInfo2,
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo2(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [20]:
# 使用ToolStrategy 支持不支持原生结构化输出的模型

from typing import Literal
from langchain.agents.structured_output import ToolStrategy

class ProductReview(BaseModel):
    """对产品评论的分析。"""
    rating: int | None = Field(description="产品的评分", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="评论的情感倾向")
    key_points: list[str] = Field(description="评论的要点。小写，每条只写2个词。")

agent = create_agent(
    model,
    response_format=ToolStrategy(ProductReview)
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]

ProductReview(rating=5, sentiment='positive', key_points=['great product', 'fast shipping'])

In [22]:
### custom tool message content
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class MeetingAction(BaseModel):
    """从会议记录中提取的行动事项。"""
    task: str = Field(description="需要完成的具体任务")
    assignee: str = Field(description="负责该任务的人员")
    priority: Literal["low", "medium", "high"] = Field(description="优先级")

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "格式存在问题。请重试。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "返回了多个结构化输出。请选择最相关的一个。"
    else:
        return f"错误: {str(error)}"

agent = create_agent(
    model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="行动事项已捕获并添加到会议记录中！",
        handle_errors=custom_error_handler
    )
)

response = agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

In [25]:
response["structured_response"]

MeetingAction(task='update the project timeline', assignee='Sarah', priority='high')

In [35]:
response["messages"]

[HumanMessage(content='From our meeting: Sarah needs to update the project timeline as soon as possible', additional_kwargs={}, response_metadata={}, id='a754a812-a84f-4dca-ba20-e20d231ce35b'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 396, 'total_tokens': 421, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'doubao-seed-1-6-flash-250828', 'system_fingerprint': None, 'id': '0217693420276934fe949375ac26682761f342713cef33224f301', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bf500-c722-7643-9392-54df083300f5-0', tool_calls=[{'name': 'MeetingAction', 'args': {'task': 'update the project timeline', 'assignee': 'Sarah', 'priority': 'high'}, 'i